In [1]:
!pip install /home/remi/py_event_studies

Processing /home/remi/py_event_studies
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for py_event_studies: filename=py_event_studies-0.2.0-py3-none-any.whl size=45490 sha256=2ee89df705e71adba76503f1849c3ed15d4f224f0f7ff52e8b46b83b9c9f2334
  Stored in directory: /tmp/pip-ephem-wheel-cache-o3s474j7/wheels/22/02/d9/e18517984e20a6f819988550f1b1e3268cc626948300fdedec
Successfully built py_event_studies
  Attempting uninstall: py_event_studies
    Found existing installation: py_event_studies 0.2.0
    Uninstalling py_event_studies-0.2.0:
      Successfully uninstalled py_event_studies-0.2.0

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import os
import time
import py_event_studies as pes
from IPython.display import display

In [3]:
pes.clear_cache() # If you want to clear the cache manually (not necessary, used here for demo and showing usefullness bellows)

Cache cleared successfully.


In [4]:
pes.load_data('../.data/CRSPAllClean.parquet')
pes.load_ff_factors('../.data/FF5.csv')

Loading and preprocessing data from ../.data/CRSPAllClean.parquet
Cached preprocessed data for ../.data/CRSPAllClean.parquet
Using cached data for ../.data/CRSPAllClean.parquet


In [6]:
num_simulation = 1000
num_stock_ptf = 50

valid_dates = pes.api.get_valid_dates()[pes.config.estim_period*2:-pes.config.event_period]
date_and_portfolio = []
while len(date_and_portfolio) < num_simulation:
    event_date = np.random.choice(valid_dates)
    valid_permnos = pes.get_valid_permno_at_date(event_date)
    if len(valid_permnos) >= num_stock_ptf:
        selected_permnos = np.random.choice(valid_permnos, num_stock_ptf, replace=False)
        date_and_portfolio.append([event_date, selected_permnos])

In [9]:
import numpy as np
import pandas as pd
from scipy import stats
import os
import time
import py_event_studies as pes

def run_power_analysis(date_and_portfolio, output_dir="power_analysis"):
    """
    Run power analysis and generate formatted tables and dataframes.
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"{output_dir}/latex", exist_ok=True)
    os.makedirs(f"{output_dir}/csv", exist_ok=True)
    
    # Define parameters
    shock_values = [0, -0.05, -0.02, -0.01, -0.005, 0.005, 0.01, 0.02, 0.05]
    shock_labels = [f"{s:.1%}" for s in shock_values]
    
    # Define gamma functions for variance multiplication
    gamma_configs = [
        {"func": lambda shape: np.ones(shape), "label": "γ=1"},
        {"func": lambda shape: np.random.uniform(1, 2, shape), "label": "γ~U[1,2]"},
        {"func": lambda shape: np.random.uniform(1.5, 2.5, shape), "label": "γ~U[1.5,2.5]"},
        {"func": lambda shape: np.random.uniform(2.5, 3.5, shape), "label": "γ~U[2.5,3.5]"}
    ]
    
    # Test statistics to evaluate
    test_types = ["std", "CS", "BMP", "KP"]
    
    # Get current cluster configuration
    cluster_nums = pes.config.cluster_num_list
    print(f"Using clusters: {cluster_nums}")
    
    # Dictionary to map cluster numbers to their indices
    cluster_to_idx = {c_num: idx for idx, c_num in enumerate(cluster_nums)}
    
    # Define models
    base_models = ["Market Model", "FF3", "FF5"]
    cluster_models = []
    
    # Generate cluster models dynamically based on current config
    for c_num in cluster_nums:
        cluster_models.extend([
            f"{c_num} clusters",
            f"{c_num} clusters + FF3", 
            f"{c_num} clusters + FF5"
        ])
    
    all_models = base_models + cluster_models
    num_simulations = len(date_and_portfolio)
    
    # Initialize rejection counters and McNemar pairs
    # Structure: {test_type: {gamma_label: {shock_label: {model: count}}}}
    rejection_counts = {}
    mcnemar_pairs = {}
    
    for test in test_types:
        rejection_counts[test] = {}
        mcnemar_pairs[test] = {}
        
        for gamma in gamma_configs:
            g_label = gamma["label"]
            rejection_counts[test][g_label] = {}
            mcnemar_pairs[test][g_label] = {}
            
            for s_label in shock_labels:
                rejection_counts[test][g_label][s_label] = {model: 0 for model in all_models}
                mcnemar_pairs[test][g_label][s_label] = {model: [] for model in all_models if model != "FF5"}
    
    # Process each simulation
    for sim_idx, (event_date, selected_permnos) in enumerate(date_and_portfolio):
        print(f"Processing simulation {sim_idx+1}/{num_simulations} - Date: {event_date}")

        # Compute baseline results
        baseline_results = pes.compute(event_date, selected_permnos)
        
        # For each gamma value (variance multiplier)
        for gamma in gamma_configs:
            gamma_label = gamma["label"]
            
            # For each shock value
            for s_idx, shock in enumerate(shock_values):
                shock_label = shock_labels[s_idx]
                
                # Create shocked results
                shocked_results = baseline_results.create_shocked_results(
                    shock=shock, 
                    c_vector_generator=gamma["func"]
                )
                
                # Process each test type
                for test in test_types:
                    # Get p-values for this test
                    p_values = getattr(shocked_results, f"{test.lower()}_p_values")
                    
                    # Check FF5 rejection
                    ff5_rejected = p_values.iloc[0, 6] < 0.05
                    
                    # Process standard models
                    if p_values.iloc[0, 4] < 0.05:  # Market Model (index 4)
                        rejection_counts[test][gamma_label][shock_label]["Market Model"] += 1
                        mcnemar_pairs[test][gamma_label][shock_label]["Market Model"].append((ff5_rejected, True))
                    else:
                        mcnemar_pairs[test][gamma_label][shock_label]["Market Model"].append((ff5_rejected, False))
                        
                    if p_values.iloc[0, 5] < 0.05:  # FF3 (index 5)
                        rejection_counts[test][gamma_label][shock_label]["FF3"] += 1
                        mcnemar_pairs[test][gamma_label][shock_label]["FF3"].append((ff5_rejected, True))
                    else:
                        mcnemar_pairs[test][gamma_label][shock_label]["FF3"].append((ff5_rejected, False))
                        
                    if ff5_rejected:  # FF5 (index 6)
                        rejection_counts[test][gamma_label][shock_label]["FF5"] += 1
                    
                    # Process cluster models
                    for c_num in cluster_nums:
                        c_idx = cluster_to_idx[c_num]
                        
                        # Cluster only (index 0)
                        model_name = f"{c_num} clusters"
                        if p_values.iloc[c_idx, 0] < 0.05:
                            rejection_counts[test][gamma_label][shock_label][model_name] += 1
                            mcnemar_pairs[test][gamma_label][shock_label][model_name].append((ff5_rejected, True))
                        else:
                            mcnemar_pairs[test][gamma_label][shock_label][model_name].append((ff5_rejected, False))
                        
                        # Cluster + FF3 (index 2)
                        model_name = f"{c_num} clusters + FF3"
                        if p_values.iloc[c_idx, 2] < 0.05:
                            rejection_counts[test][gamma_label][shock_label][model_name] += 1
                            mcnemar_pairs[test][gamma_label][shock_label][model_name].append((ff5_rejected, True))
                        else:
                            mcnemar_pairs[test][gamma_label][shock_label][model_name].append((ff5_rejected, False))
                        
                        # Cluster + FF5 (index 3)
                        model_name = f"{c_num} clusters + FF5"
                        if p_values.iloc[c_idx, 3] < 0.05:
                            rejection_counts[test][gamma_label][shock_label][model_name] += 1
                            mcnemar_pairs[test][gamma_label][shock_label][model_name].append((ff5_rejected, True))
                        else:
                            mcnemar_pairs[test][gamma_label][shock_label][model_name].append((ff5_rejected, False))
            

    # Calculate rejection rates and create result dataframes
    results = {}
    
    for test in test_types:
        results[test] = {}
        
        for gamma in gamma_configs:
            gamma_label = gamma["label"]
            
            # Create dataframe for this test and gamma
            df = pd.DataFrame(index=all_models, columns=shock_labels)
            
            # Calculate rejection rates
            for shock_label in shock_labels:
                for model in all_models:
                    rejection_rate = rejection_counts[test][gamma_label][shock_label][model] / num_simulations * 100
                    df.loc[model, shock_label] = rejection_rate
            
            results[test][gamma_label] = df
            
            # Save CSV
            csv_path = f"{output_dir}/csv/power_{test}_{gamma_label.replace('=', '').replace('~', '').replace('[', '').replace(']', '').replace(',', '_')}.csv"
            df.to_csv(csv_path)
            
            # Create LaTeX table
            create_power_table(
                test, 
                gamma_label, 
                base_models,
                cluster_models,
                df, 
                mcnemar_pairs[test][gamma_label], 
                num_simulations,
                output_dir
            )
    
    print(f"Analysis complete. Results saved to {output_dir}/")
    return results

def perform_mcnemar_test(pairs, num_simulations):
    """Calculate McNemar's test to determine if a model is better than FF5."""
    if not pairs or len(pairs) < num_simulations * 0.8:  # Need sufficient samples
        return False
        
    # Count contingency table entries
    n11 = sum(1 for x, y in pairs if x and y)      # Both reject
    n12 = sum(1 for x, y in pairs if x and not y)  # FF5 rejects, model doesn't
    n21 = sum(1 for x, y in pairs if not x and y)  # FF5 doesn't, model rejects
    n22 = sum(1 for x, y in pairs if not x and not y)  # Neither rejects
    
    # Skip if no or few disagreements
    if n12 + n21 < 5:
        return False
    
    # Calculate McNemar's statistic with continuity correction
    statistic = ((abs(n12 - n21) - 1)**2) / (n12 + n21)
    p_value = 1 - stats.chi2.cdf(statistic, 1)
    
    # Is the model significantly better? (one-sided test)
    is_better = (p_value < 0.05) and (n21 > n12)
    
    return is_better

def create_power_table(test_type, gamma_label, base_models, cluster_models, 
                      df, mcnemar_pairs, num_simulations, output_dir):
    """Create a LaTeX table for a specific test and gamma value."""
    
    # Create table header
    table = "\\begin{table}[!h]\n"
    table += f"    \\caption{{Type II error rates (power): {gamma_label}}}\n"
    table += "    \\small\n"
    table += "    \\begin{center}\n"
    table += "    \\begin{tabular}{l" + "r" * len(df.columns) + "}\n"
    table += "    \\hline\n"
    table += "\\textit{Abnormal return ($\\delta)$} &  "
    for col in df.columns:
        col_formatted = col.replace('%', '\\%')
        table += f" & \\textit{{{col_formatted}}}"
    table += "\\T\\B\\\\\n"
    table += "    \\hline\n"
        
    # Add base models
    for model in base_models:
        table += f"{model}\t"
        for col in df.columns:
            rate = df.loc[model, col]
            table += f"& {rate:.1f}\\% "
        table += "\\\\\n"
    
    # Add divider
    table += "\\hline\n"
    
    # Add cluster models
    for model in cluster_models:
        table += f"{model}\t"
        for col in df.columns:
            rate = df.loc[model, col]
            
            # Determine if model is significantly better than FF5
            is_better = False
            if col != df.columns[0]:  # Skip the 0 shock value
                pairs = mcnemar_pairs[col].get(model, [])
                is_better = perform_mcnemar_test(pairs, num_simulations)
            
            if is_better:
                table += f"& \\textbf{{{rate:.1f}\\%}} "
            else:
                table += f"& {rate:.1f}\\% "
        table += "\\\\\n"
    
    # Add footer
    table += "\\hline\\\\\n"
    table += "\\multicolumn{" + str(len(df.columns) + 1) + "}{l}{\\itshape Bold face indicates rejection of power equal to FF5 model at the 5\\% level based on McNemar's test}\n"
    table += "    \\end{tabular}\n"
    table += "    \\end{center}\n"
    table += "\\end{table}\n"
    
    # Save the table
    clean_label = gamma_label.replace('=', '').replace('~', '').replace('[', '').replace(']', '').replace(',', '_').replace('.', '_')
    filename = f"{output_dir}/latex/power_table_{test_type}_{clean_label}.tex"
    with open(filename, "w") as f:
        f.write(table)
    
    print(f"Created LaTeX table: {filename}")

# Example of how to run the analysis
# pes.config.cluster_num_list = [15, 45, 50]  # Set desired clusters
# results_df = run_power_analysis(date_and_portfolio)

# Example of how to run the analysis
# pes.config.cluster_num_list = [15, 45, 50]  # Set desired clusters
# results_df = run_power_analysis(date_and_portfolio)
# Configure the clusters you want to analyze
pes.config.cluster_num_list = [5, 10, 15, 20, 25, 30, 35, 40, 50]

# Run the analysis
results_df = run_power_analysis(date_and_portfolio)

# Access specific results (example)
cs_gamma1_results = results_df["CS"]["γ=1"]
print(cs_gamma1_results)

Using clusters: [5, 10, 15, 20, 25, 30, 35, 40, 50]
Processing simulation 1/1000 - Date: 20010831
Processing simulation 2/1000 - Date: 19760225
Processing simulation 3/1000 - Date: 19820329
Processing simulation 4/1000 - Date: 19910717
Processing simulation 5/1000 - Date: 19841011
Processing simulation 6/1000 - Date: 20220516
Processing simulation 7/1000 - Date: 20190514
Processing simulation 8/1000 - Date: 19970625
Processing simulation 9/1000 - Date: 19840315
Processing simulation 10/1000 - Date: 19831205
Processing simulation 11/1000 - Date: 19910510
Processing simulation 12/1000 - Date: 20141224
Processing simulation 13/1000 - Date: 20160411
Processing simulation 14/1000 - Date: 19940801
Processing simulation 15/1000 - Date: 20020710
Processing simulation 16/1000 - Date: 19861204
Processing simulation 17/1000 - Date: 19741223
Processing simulation 18/1000 - Date: 19950703
Processing simulation 19/1000 - Date: 20051129
Processing simulation 20/1000 - Date: 19980205
Processing simula

/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:140: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: divide by zero encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/_core/_m

Processing simulation 161/1000 - Date: 20181024
Processing simulation 162/1000 - Date: 20040714
Processing simulation 163/1000 - Date: 20050615
Processing simulation 164/1000 - Date: 20170103
Processing simulation 165/1000 - Date: 19920312
Processing simulation 166/1000 - Date: 20231113
Processing simulation 167/1000 - Date: 20180801
Processing simulation 168/1000 - Date: 19990303
Processing simulation 169/1000 - Date: 20140908
Processing simulation 170/1000 - Date: 19980306
Processing simulation 171/1000 - Date: 19880315
Processing simulation 172/1000 - Date: 19960118
Processing simulation 173/1000 - Date: 20091008
Processing simulation 174/1000 - Date: 19841115
Processing simulation 175/1000 - Date: 19810407
Processing simulation 176/1000 - Date: 20070205
Processing simulation 177/1000 - Date: 19780105
Processing simulation 178/1000 - Date: 19880106
Processing simulation 179/1000 - Date: 19760202
Processing simulation 180/1000 - Date: 20210715
Processing simulation 181/1000 - Date: 1

/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:140: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: divide by zero encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/_core/_m

Processing simulation 434/1000 - Date: 19901016
Processing simulation 435/1000 - Date: 20151006
Processing simulation 436/1000 - Date: 20111019
Processing simulation 437/1000 - Date: 19831201
Processing simulation 438/1000 - Date: 20000515
Processing simulation 439/1000 - Date: 19790920
Processing simulation 440/1000 - Date: 19881220
Processing simulation 441/1000 - Date: 20071206
Processing simulation 442/1000 - Date: 19750918
Processing simulation 443/1000 - Date: 19890103
Processing simulation 444/1000 - Date: 19890419
Processing simulation 445/1000 - Date: 19971124
Processing simulation 446/1000 - Date: 19880914
Processing simulation 447/1000 - Date: 19800318
Processing simulation 448/1000 - Date: 20060622
Processing simulation 449/1000 - Date: 20180906
Processing simulation 450/1000 - Date: 20050331
Processing simulation 451/1000 - Date: 19901115
Processing simulation 452/1000 - Date: 20180123
Processing simulation 453/1000 - Date: 20220609
Processing simulation 454/1000 - Date: 1

/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:140: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: divide by zero encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/_core/_m

Processing simulation 455/1000 - Date: 20161014
Processing simulation 456/1000 - Date: 20140916
Processing simulation 457/1000 - Date: 19921120
Processing simulation 458/1000 - Date: 19750826
Processing simulation 459/1000 - Date: 20090924
Processing simulation 460/1000 - Date: 19760319
Processing simulation 461/1000 - Date: 20030206
Processing simulation 462/1000 - Date: 19911011
Processing simulation 463/1000 - Date: 20141008
Processing simulation 464/1000 - Date: 20030311
Processing simulation 465/1000 - Date: 20160526
Processing simulation 466/1000 - Date: 20200616
Processing simulation 467/1000 - Date: 19860716
Processing simulation 468/1000 - Date: 19870826
Processing simulation 469/1000 - Date: 20100816
Processing simulation 470/1000 - Date: 19881024
Processing simulation 471/1000 - Date: 19850409
Processing simulation 472/1000 - Date: 20120319
Processing simulation 473/1000 - Date: 20180604
Processing simulation 474/1000 - Date: 19781024
Processing simulation 475/1000 - Date: 2

/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:140: RuntimeWarning: invalid value encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/py_event_studies/_ar_statistic_tests.py:83: RuntimeWarning: divide by zero encountered in divide
  event_residual_bar = event_residual / sigma_adj
/pyenv/versions/3.10.16/lib/python3.10/site-packages/numpy/_core/_m

Processing simulation 839/1000 - Date: 19960425
Processing simulation 840/1000 - Date: 20000711
Processing simulation 841/1000 - Date: 20141014
Processing simulation 842/1000 - Date: 19871105
Processing simulation 843/1000 - Date: 20021105
Processing simulation 844/1000 - Date: 20000131
Processing simulation 845/1000 - Date: 20000504
Processing simulation 846/1000 - Date: 20151106
Processing simulation 847/1000 - Date: 19841108
Processing simulation 848/1000 - Date: 20180123
Processing simulation 849/1000 - Date: 19800616
Processing simulation 850/1000 - Date: 20201211
Processing simulation 851/1000 - Date: 20150624
Processing simulation 852/1000 - Date: 19950828
Processing simulation 853/1000 - Date: 20161003
Processing simulation 854/1000 - Date: 19801223
Processing simulation 855/1000 - Date: 20161017
Processing simulation 856/1000 - Date: 19840216
Processing simulation 857/1000 - Date: 19911119
Processing simulation 858/1000 - Date: 19790828
Processing simulation 859/1000 - Date: 1

In [10]:
results_df['KP']['γ=1']

,0.0%,-5.0%,-2.0%,-1.0%,-0.5%,0.5%,1.0%,2.0%,5.0%
Market Model,6.0,99.5,95.1,69.0,29.7,26.0,67.8,96.6,99.5
FF3,4.9,99.6,98.3,85.7,43.3,37.9,84.1,98.5,99.6
FF5,4.9,99.6,98.3,85.7,42.8,39.3,84.2,98.6,99.6
5 clusters,5.2,99.6,98.5,86.4,44.1,39.6,84.6,98.7,99.6
5 clusters + FF3,5.7,99.6,98.6,87.4,45.9,40.7,85.4,98.9,99.6
5 clusters + FF5,4.7,99.6,98.6,87.8,45.8,40.7,85.6,98.8,99.6
10 clusters,5.0,99.6,98.7,87.4,44.8,39.6,84.9,99.0,99.6
10 clusters + FF3,5.2,99.6,98.6,88.0,46.2,41.5,85.1,98.9,99.6
10 clusters + FF5,5.1,99.6,98.6,88.0,45.7,41.7,85.4,98.8,99.6
15 clusters,5.2,99.6,98.6,87.9,45.5,40.9,85.5,98.9,99.6


In [11]:
results_df['CS']['γ=1']

,0.0%,-5.0%,-2.0%,-1.0%,-0.5%,0.5%,1.0%,2.0%,5.0%
Market Model,13.2,99.8,93.1,66.4,34.0,30.2,62.9,95.0,99.9
FF3,5.2,99.7,94.9,70.3,30.8,24.3,64.6,96.1,99.9
FF5,5.2,99.7,94.7,69.8,30.6,25.0,64.4,96.0,99.9
5 clusters,4.7,99.7,94.8,70.7,31.6,25.2,65.4,96.5,100.0
5 clusters + FF3,4.2,99.7,94.5,70.7,30.4,23.8,65.4,96.2,100.0
5 clusters + FF5,4.0,99.7,94.7,70.1,30.5,23.6,65.2,96.1,100.0
10 clusters,4.7,99.7,94.9,71.2,31.4,24.7,65.2,96.6,100.0
10 clusters + FF3,4.6,99.7,94.9,70.8,30.4,23.4,65.4,96.6,100.0
10 clusters + FF5,4.2,99.7,94.8,69.9,30.7,23.7,65.0,96.4,100.0
15 clusters,4.5,99.7,95.0,70.5,31.7,24.8,65.3,96.9,100.0
